<a href="https://colab.research.google.com/github/obeabi/ProjectPortfolio/blob/master/leadership_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [99]:
import yfinance as yf
import gradio as gr
print(yf.__version__)
import pandas as pd
import numpy as np
from datetime import datetime, timezone
datetime.now(timezone.utc)
import time
import plotly.express as px
import random
import requests
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Optional
import warnings
warnings.filterwarnings("ignore")
# ensure reproducibility
random.seed(42)

# Display all rows
pd.set_option('display.max_rows', None)

# Display all columns
pd.set_option('display.max_columns', None)

# Prevent wrapping of wide DataFrames
pd.set_option('display.expand_frame_repr', False)

# Display full content of each cell
pd.set_option('display.max_colwidth', None)

# Display the full width of the DataFrame
pd.set_option('display.width', None)
print("Libraries Installed!")

0.2.66
Libraries Installed!


In [102]:
class LeadershipEngine:
    def __init__(self, universe_csv: str):
        self.universe = self.load_universe(universe_csv)
        self.data = None
        self.stock_df = None
        self.sector_df = None
        self.mode = None  # "intraday" or "daily"

    # ---------------------------------------------------------
    # MODULE 1 — UNIVERSE LOADER
    # ---------------------------------------------------------
    def load_universe(self, csv_path: str) -> pd.DataFrame:
        df = pd.read_csv(csv_path)
        df.columns = [c.strip() for c in df.columns]

        required = {"Ticker", "Company", "Sector", "SectorETF"}
        if not required.issubset(df.columns):
            raise ValueError(f"CSV must contain columns: {required}")

        df["Ticker"] = (
            df["Ticker"]
            .str.upper()
            .str.strip()
            .str.replace(".", "-", regex=False)
        )

        df["SectorETF"] = (
            df["SectorETF"]
            .str.upper()
            .str.strip()
            .str.replace(".", "-", regex=False)
        )

        return df

    # ---------------------------------------------------------
    # MODULE 2 — DAILY DATA (BASELINE)
    # ---------------------------------------------------------
    def download_daily(self, batch_size=100):
        tickers = list(
            dict.fromkeys(
                ["SPY"]
                + self.universe["SectorETF"].unique().tolist()
                + self.universe["Ticker"].tolist()
            )
        )

        frames = []

        for i in range(0, len(tickers), batch_size):
            batch = tickers[i:i + batch_size]

            try:
                data = yf.download(
                    tickers=batch,
                    period="6mo",
                    interval="1d",
                    auto_adjust=True,
                    group_by="ticker",
                    threads=True,
                    progress=False,
                )

                if not data.empty:
                    frames.append(data)

            except Exception:
                continue

        if len(frames) == 0:
            raise RuntimeError("Daily data unavailable — universe may contain invalid tickers.")

        combined = pd.concat(frames, axis=1)

        if isinstance(combined.columns, pd.MultiIndex):
            combined = combined.sort_index(axis=1)

        self.data = combined
        self.mode = "daily"

    # ---------------------------------------------------------
    # MODULE 3 — INTRADAY (OPTIONAL OVERRIDE)
    # ---------------------------------------------------------
    def download_intraday(self, batch_size=100):
        tickers = list(
            dict.fromkeys(
                ["SPY"]
                + self.universe["SectorETF"].unique().tolist()
                + self.universe["Ticker"].tolist()
            )
        )

        frames = []

        for i in range(0, len(tickers), batch_size):
            batch = tickers[i:i + batch_size]

            try:
                data = yf.download(
                    tickers=batch,
                    period="5d",
                    interval="5m",
                    auto_adjust=True,
                    group_by="ticker",
                    threads=True,
                    progress=False,
                    prepost=False,
                )
                if not data.empty:
                    frames.append(data)
            except Exception:
                continue

        if len(frames) == 0:
            raise RuntimeError("Intraday unavailable.")

        combined = pd.concat(frames, axis=1)

        if isinstance(combined.columns, pd.MultiIndex):
            combined = combined.sort_index(axis=1)

        self.data = combined
        self.mode = "intraday"

    # ---------------------------------------------------------
    # MODULE 4 — SAFE DIVISION & RETURNS
    # ---------------------------------------------------------
    def safe_div(self, a, b):
        if pd.isna(a) or pd.isna(b) or b == 0:
            return np.nan
        return a / b - 1

    def calc_returns_intraday(self, series: pd.Series):
        series = series.dropna()
        n = len(series)

        if n < 2:
            return np.nan, np.nan, np.nan

        latest = series.iloc[-1]
        r5 = self.safe_div(latest, series.iloc[-2] if n >= 2 else np.nan)
        r1h = self.safe_div(latest, series.iloc[-12] if n >= 12 else np.nan)

        today = series.index[-1].date()
        prev_day = series[series.index.date < today]
        prev_close = prev_day.iloc[-1] if len(prev_day) else np.nan

        r1d = self.safe_div(latest, prev_close)

        return r5, r1h, r1d

    def calc_returns_daily(self, series: pd.Series):
        series = series.dropna()
        n = len(series)

        if n < 2:
            return np.nan, np.nan, np.nan

        latest = series.iloc[-1]
        prev_close = series.iloc[-2]

        r1d = self.safe_div(latest, prev_close)
        return np.nan, np.nan, r1d

    # ---------------------------------------------------------
    # MODULE 5 — CLOSE EXTRACTOR
    # ---------------------------------------------------------
    def get_close(self, ticker: str) -> pd.Series:
        try:
            if isinstance(self.data.columns, pd.MultiIndex):
                if ticker in self.data.columns.get_level_values(0):
                    return self.data[ticker]["Close"]
            if "Close" in self.data.columns:
                return self.data["Close"]
        except Exception:
            pass
        return pd.Series(dtype=float)

    # ---------------------------------------------------------
    # MODULE 6 — STOCK RS
    # ---------------------------------------------------------
    def compute_stock_rs(self):
        rows = []

        spy_close = self.get_close("SPY")
        if spy_close.dropna().empty:
            raise RuntimeError("SPY data unavailable.")

        if self.mode == "intraday":
            spy_r5, spy_r1h, spy_r1d = self.calc_returns_intraday(spy_close)
        else:
            spy_r5, spy_r1h, spy_r1d = self.calc_returns_daily(spy_close)

        sector_returns = {}
        for _, row in self.universe[["Sector", "SectorETF"]].drop_duplicates().iterrows():
            etf_close = self.get_close(row["SectorETF"])
            if etf_close.dropna().empty:
                continue

            if self.mode == "intraday":
                sr5, sr1h, sr1d = self.calc_returns_intraday(etf_close)
            else:
                sr5, sr1h, sr1d = self.calc_returns_daily(etf_close)

            sector_returns[row["Sector"]] = (sr5, sr1h, sr1d)

        for _, stock in self.universe.iterrows():
            ticker = stock["Ticker"]
            sector = stock["Sector"]

            close = self.get_close(ticker)
            if close.dropna().empty:
                continue

            if self.mode == "intraday":
                r5, r1h, r1d = self.calc_returns_intraday(close)
            else:
                r5, r1h, r1d = self.calc_returns_daily(close)

            sr5, sr1h, sr1d = sector_returns.get(sector, (np.nan, np.nan, np.nan))

            rows.append({
                "Ticker": ticker,
                "Company": stock["Company"],
                "Sector": sector,
                "R5": r5,
                "R1H": r1h,
                "R1D": r1d,
                "RS5_SPY": r5 - spy_r5,
                "RS1H_SPY": r1h - spy_r1h,
                "RS1D_SPY": r1d - spy_r1d,
                "RS5_SEC": r5 - sr5,
                "RS1H_SEC": r1h - sr1h,
                "RS1D_SEC": r1d - sr1d,
            })

        df = pd.DataFrame(rows)

        for col in ["RS5_SPY", "RS1H_SPY", "RS1D_SPY", "RS5_SEC", "RS1H_SEC", "RS1D_SEC"]:
            std = df[col].std(ddof=0)
            df[f"Z_{col}"] = 0 if std == 0 or pd.isna(std) else (df[col] - df[col].mean()) / std

        self.stock_df = df

    # ---------------------------------------------------------
    # MODULE 7 — SECTOR STRENGTH
    # ---------------------------------------------------------
    def compute_sector_strength(self):
        rows = []

        spy_close = self.get_close("SPY")
        if spy_close.dropna().empty:
            raise RuntimeError("SPY data unavailable.")

        if self.mode == "intraday":
            _, _, spy_r1d = self.calc_returns_intraday(spy_close)
        else:
            _, _, spy_r1d = self.calc_returns_daily(spy_close)

        for _, row in self.universe[["Sector", "SectorETF"]].drop_duplicates().iterrows():
            etf_close = self.get_close(row["SectorETF"])
            if etf_close.dropna().empty:
                continue

            if self.mode == "intraday":
                _, _, r1d = self.calc_returns_intraday(etf_close)
            else:
                _, _, r1d = self.calc_returns_daily(etf_close)

            rows.append({
                "Sector": row["Sector"],
                "SectorETF": row["SectorETF"],
                "Sector1D": r1d,
                "SectorRS_SPY": r1d - spy_r1d,
            })

        df = pd.DataFrame(rows)

        if df.empty:
            sector_map = self.universe[["Sector", "SectorETF"]].drop_duplicates()
            df = sector_map.copy()
            df["Sector1D"] = np.nan
            df["SectorRS_SPY"] = np.nan

        std = df["SectorRS_SPY"].std(ddof=0)
        df["Z_Sector"] = 0 if std == 0 or pd.isna(std) else (
            df["SectorRS_SPY"] - df["SectorRS_SPY"].mean()
        ) / std

        self.sector_df = df

    # ---------------------------------------------------------
    # MODULE 8 — LEADERSHIP SCORE
    # ---------------------------------------------------------
    def compute_leadership_score(self):
        df = self.stock_df.merge(
            self.sector_df[["Sector", "SectorRS_SPY", "Z_Sector"]],
            on="Sector",
            how="left",
        )

        df["Score"] = (
            df["Z_RS1D_SPY"] * 0.35 +
            df["Z_RS1D_SEC"] * 0.25 +
            df["Z_Sector"] * 0.20 +
            df["Z_RS1H_SPY"] * 0.15 +
            df["Z_RS5_SPY"] * 0.05
        )

        df["Percentile"] = df["Score"].rank(pct=True) * 100

        self.stock_df = df

    # ---------------------------------------------------------
    # MODULE 9 — LEADERSHIP TYPE
    # ---------------------------------------------------------
    def compute_leadership_type(self):
        df = self.stock_df

        df["Type"] = np.where(
            (df["RS1D_SPY"] > 0) &
            (df["RS1D_SEC"] > 0) &
            (df["SectorRS_SPY"] > 0),
            "A-Line",
            np.where(
                (df["RS1D_SPY"] > 0) & (df["RS1D_SEC"] <= 0),
                "Market Leader",
                np.where(
                    (df["RS1D_SPY"] <= 0) & (df["RS1D_SEC"] > 0),
                    "Sector Leader",
                    "Weak",
                ),
            ),
        )

        self.stock_df = df

    # ---------------------------------------------------------
    # MODULE 10 — WATCHLIST
    # ---------------------------------------------------------
    def load_watchlist(self, watchlist_csv):
        df = pd.read_csv(watchlist_csv)
        df.columns = [c.strip() for c in df.columns]

        ticker_col = "Ticker" if "Ticker" in df.columns else df.columns[0]

        df[ticker_col] = (
            df[ticker_col]
            .str.upper()
            .str.strip()
            .str.replace(".", "-", regex=False)
        )

        self.watchlist = df[ticker_col].unique().tolist()

    def watchlist_matches(self):
        if not hasattr(self, "watchlist"):
            raise RuntimeError("Watchlist not loaded.")

        df = self.stock_df.copy()
        return df[df["Ticker"].isin(self.watchlist)].sort_values("Score", ascending=False)

    def watchlist_intersection(self, percentile_threshold=80):
        if not hasattr(self, "watchlist"):
            raise RuntimeError("Watchlist not loaded.")

        df = self.stock_df.copy()
        leaders = df[df["Percentile"] >= percentile_threshold]
        return leaders[leaders["Ticker"].isin(self.watchlist)].sort_values("Score", ascending=False)

    # ---------------------------------------------------------
    # MODULE 11 — PIPELINE
    # ---------------------------------------------------------
    def run(self):
        # Always load daily baseline
        self.download_daily()

        # Try intraday override
        try:
            self.download_intraday()
        except Exception:
            pass  # keep daily

        # Continue pipeline
        self.compute_stock_rs()
        self.compute_sector_strength()
        self.compute_leadership_score()
        self.compute_leadership_type()

    # ---------------------------------------------------------
    # MODULE 12 — REPORT
    # ---------------------------------------------------------
    def report(self) -> pd.DataFrame:
      df = self.stock_df.copy()

      # Remove weak stocks
      #df = df[df["Type"] != "Weak"]

      if self.mode == "intraday":
        print("🟢 Using intraday 5m data.")
      else:
        print("⚠️ Using daily data.")

      return df.sort_values("Score", ascending=False)



In [109]:
if __name__ == "__main__":
    engine = LeadershipEngine("sp50_universe.csv")
    engine.run()
    #df = engine.report()
    engine.load_watchlist("my_watchlist.csv")
    #print(df.head(20))
    #print("\n=== WATCHLIST LOADED ===")
    #print(engine.watchlist)
    wl = engine.watchlist_matches()
    #print("\n=== WATCHLIST LEADERSHIP RANKING ===")
    print(wl[["Ticker", "Company", "Sector", "Type", "Score", "Percentile"]].head(20))
    inter = engine.watchlist_intersection(percentile_threshold=80)
    #print("\n=== INTERSECTION WITH TOP LEADERS (>=80th Percentile) ===")
    #print(inter[["Ticker", "Company", "Sector", "Type", "Score", "Percentile"]].head(20))
    #inter = df.copy()



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 393, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2280, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 1657, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/anyio/to_thread.py", line 65, in run_sync
    return await get_async_backend().run_sync_in_worker_thread(
           ^^^^^

🟢 Using intraday 5m data.
   Ticker               Company       Sector  Type     Score  Percentile
43   AXON       Axon Enterprise  Industrials  Weak  2.154117  100.000000
25    AMP  Ameriprise Financial   Financials  Weak -0.586779   20.454545


In [92]:
#inter[["Ticker", "Company", "Sector", "Type", "Score", "Percentile"]].head()
#inter[["Ticker", "Company", "Sector","Z_RS5_SPY",'Z_RS1H_SPY', 'Z_RS1D_SPY', 'Z_RS5_SEC', 'Z_RS1H_SEC', 'Z_RS1D_SEC','Z_Sector',
                 # "Type", "Score", "Percentile"]].head()

In [93]:
#def show_leaders():
    #return inter[["Ticker", "Company", "Sector","Z_RS5_SPY",'Z_RS1H_SPY', 'Z_RS1D_SPY', 'Z_RS5_SEC', 'Z_RS1H_SEC', 'Z_RS1D_SEC','Z_Sector',
                  #"Type", "Score", "Percentile"]]

#gr.Interface(fn=show_leaders, inputs=None, outputs="dataframe").launch()


In [106]:
def show_leaders():
    # IMPORTANT: recompute engine each refresh
    engine.run()
    inter = engine.watchlist_intersection(percentile_threshold=80)
    # Remove weak stocks
    inter = inter[inter["Type"] != "Weak"]

    return inter[[
        "Ticker", "Company", "Sector",
        "Z_RS5_SPY", "Z_RS1H_SPY", "Z_RS1D_SPY",
        "Z_RS5_SEC", "Z_RS1H_SEC", "Z_RS1D_SEC",
        "Z_Sector",
        "Type", "Score", "Percentile"
    ]]

with gr.Blocks() as app:
    gr.Markdown("## 📈 A‑Line Leaders (Auto‑Refresh Every 5 Minutes)")

    output_df = gr.Dataframe(interactive=False)

    # Auto-refresh every 300 seconds
    timer = gr.Timer(300)
    timer.tick(
        fn=show_leaders,
        inputs=None,
        outputs=output_df
    )

app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1e699cc4165278813a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [108]:


# ---------------------------------------------------------
# INIT ENGINE
# ---------------------------------------------------------
engine = LeadershipEngine("sp50_universe.csv")
engine.load_watchlist("my_watchlist.csv")


# ---------------------------------------------------------
# PLOT HELPERS
# ---------------------------------------------------------
def momentum_heatmap(df: pd.DataFrame):
    cols = ["Ticker", "RS5_SPY", "RS1H_SPY", "RS1D_SPY", "SectorRS_SPY", "Score"]
    available = [c for c in cols if c in df.columns]

    if len(available) <= 1:
        return px.imshow([[0]], title="No momentum data")

    heat = df[available].set_index("Ticker")

    fig = px.imshow(
        heat,
        color_continuous_scale="RdYlGn",
        aspect="auto",
        title="Momentum Heatmap (RS vs SPY / Sector / Score)",
    )
    return fig


def sector_rotation_chart(sector_df: pd.DataFrame):
    if sector_df is None or sector_df.empty:
        return px.bar(title="No sector data")

    fig = px.bar(
        sector_df,
        x="Sector",
        y="SectorRS_SPY",
        color="Z_Sector",
        title="Sector Rotation (RS vs SPY)",
        color_continuous_scale="RdYlGn",
    )
    return fig


def rs_history_chart(ticker: str):
    if not ticker:
        return px.line(title="Select a ticker")

    hist = yf.Ticker(ticker).history(period="3mo", interval="1d")
    spy = yf.Ticker("SPY").history(period="3mo", interval="1d")

    if hist.empty or spy.empty:
        return px.line(title=f"No RS history for {ticker}")

    rs = hist["Close"] / spy["Close"] - 1

    fig = px.line(rs, title=f"RS Trend History — {ticker}")
    fig.update_layout(xaxis_title="Date", yaxis_title="RS vs SPY")
    return fig


# ---------------------------------------------------------
# CORE ENGINE RUNNER
# ---------------------------------------------------------
def run_engine():
    engine.run()

    df = engine.report()
    wl = engine.watchlist_matches()
    inter = engine.watchlist_intersection(percentile_threshold=80)
    # Remove weak stocks
    inter = inter[inter["Type"] != "Weak"]

    heatmap = momentum_heatmap(df)
    sector_fig = sector_rotation_chart(engine.sector_df)

    return df, wl, inter, heatmap, sector_fig


def update_ticker_choices():
    engine.run()
    df = engine.report()
    return gr.Dropdown.update(choices=df["Ticker"].tolist())


def rs_history_wrapper(ticker):
    return rs_history_chart(ticker)


# ---------------------------------------------------------
# GRADIO DASHBOARD
# ---------------------------------------------------------
with gr.Blocks() as app:
    gr.Markdown("# 📈 A-Line Leadership Scanner (Gradio Edition)")
    gr.Markdown("Intraday when available, daily baseline always. Auto-refresh every 5 minutes.")

    # ---------------- Tabs ----------------
    with gr.Tab("Leaders"):
        leaders_df = gr.Dataframe(interactive=False)

    with gr.Tab("Watchlist"):
        wl_df = gr.Dataframe(interactive=False)
        inter_df = gr.Dataframe(interactive=False)

    with gr.Tab("Momentum Heatmap"):
        heatmap_plot = gr.Plot()

    with gr.Tab("Sector Rotation"):
        sector_plot = gr.Plot()

    with gr.Tab("RS History"):
        ticker_input = gr.Dropdown(choices=[], label="Select Ticker")
        rs_plot = gr.Plot()

    # ---------------- Auto-refresh every 5 minutes ----------------
    timer = gr.Timer(5)  # 300 seconds = 5 minutes
    timer.tick(
        fn=run_engine,
        inputs=None,
        outputs=[leaders_df, wl_df, inter_df, heatmap_plot, sector_plot]
    )

    # ---------------- Populate ticker dropdown ----------------
    app.load(
        fn=update_ticker_choices,
        inputs=None,
        outputs=ticker_input,
    )

    # ---------------- RS history chart ----------------
    ticker_input.change(
        fn=rs_history_wrapper,
        inputs=ticker_input,
        outputs=rs_plot,
    )

app.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b6e4fd8afca9d46d33.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
